# Homework 9 - Trent Douglas
## Problem 1: Inclined GEO Vector (TOD)
- Propagate GEO orbit for 90,000 seconds using central body only
– Use 300 sec stepsize, 300 steps
– Compute ECEF position at each point
– Compute geodetic latitude, longitude, altitude at each point
– Plot Latitude vs. Longitude

Note: Assume epoch is UT1

#### Epoch: 2 Jan 2012 00:00:00

#### State Vector (ECEF)

| Parameter | Value            | Units  | Parameter | Value            | Units |
|----------|------------------|--------|----------|------------------|--------|
| X        | -20911545.881328 | m      | A        | 42164171.689784  | m      |
| Y        | 36157328.360915  | m      | E        | 0.001000         | —      |
| Z        | 5485907.544579   | m      | I        | 7.946886         | deg    |
| XD       | -2652.329846     | m/sec  | RAAN     | 49.847107        | deg    |
| YD       | -1553.892092     | m/sec  | WP       | 40.314013        | deg    |
| ZD       | 143.120484       | m/sec  | NU       | 30.057358        | deg    |

#### Geodetic Coordinates

| Parameter | Value       | Units |
|----------|-------------|--------|
| LATD     | 7.483069    | deg    |
| LON      | 18.992722   | deg    |
| ALTD     | 35749.89443 | km     |

In [ ]:
from standards import *
import copy

Pos_ = Vector3(-20911545.881328, 36157328.360915, 5485907.544579)
Vel_ = Vector3(-2652.329846, -1553.892092, 143.120484)

A = 42164171.689784
E = 0.001000
I = 7.946886
RAAN = 49.847107 # deg
WP = 40.314013 # deg
NU = 30.057358 # deg
LATD = 7.483069 # deg
LON = 18.992722 # deg
ALTD = 35749.89443 # km
f = 1/298.257223563
Earth_Eccentricity = sqrt(2*f-f**2)

Step_Size = 300 # sec
Step_Num = 300
Earth_rotation = 72.921151467e-6 #rad/sec
Earth_gravitational_parameter = 3.986004418e14 #m^3/s^2
Earth_radius = 6378137 #m

UTC_year = 2012
UTC_month = 1
UTC_day = 2
UTC_hour = 0
UTC_minute = 0
UTC_second = 0

start_time = datetime(
    UTC_year,
    UTC_month,
    UTC_day,
    UTC_hour,
    UTC_minute,
    UTC_second
)

class Step:
    def __init__(self, T, RK_X, RK_Y, RK_Z,RK_XD, RK_YD, RK_ZD, LAT, LON, ALTD, GAH):
        self.T = T
        self.RK_X = RK_X
        self.RK_Y = RK_Y
        self.RK_Z = RK_Z
        self.RK_XD = RK_XD
        self.RK_YD = RK_YD
        self.RK_ZD = RK_ZD
        self.LAT = LAT
        self.LON = LON
        self.ALTD = ALTD
        self.GAH = GAH

        
def rotate_TOD_to_ECEF(epoch: datetime, pos_: Vector3, earth_rotation, inverse=False) -> tuple[float, np.array]:
    j_date = compute_j_date(epoch.year, epoch.month, epoch.day, epoch.hour, epoch.minute, epoch.second)
    sec_From_J2000_to_epoch = (j_date - 2451545.0)*86400    
    t_u = (floor(sec_From_J2000_to_epoch/86400 + 0.5) - 0.5)/36525
    t = sec_From_J2000_to_epoch - t_u*36525*86400
    gah = t*earth_rotation + (24110.54841 + 8640184.812866*t_u + 0.093104*(t_u**2) - 6.2e-6*(t_u**3)) * (2*pi/86400)
    R = np.array([
        [cos(gah), sin(gah), 0],
        [-sin(gah), cos(gah), 0],
        [0, 0, 1]
    ])
    if inverse: return gah, np.linalg.inv(R) @ pos_.get_np_vector()
    return gah, R @ pos_.get_np_vector()

def calculate_lat_lon_h(Earth_Eccentricity, Pos_, Earth_Radius) -> tuple[float, float, float]:
    z_i_0 = -(Earth_Eccentricity**2)*Pos_.z
    r = Pos_.magnitude()
    diff = 1.0
    z_i = z_i_0
    while (abs(diff) > 0.000000001):
        z_i_b = Pos_.z - z_i
        sin_lat = z_i_b/r
        N_i = Earth_Radius/sqrt(1-(Earth_Eccentricity**2)*sin_lat**2)
        z_i_next = -N_i*(Earth_Eccentricity**2)*sin_lat
        diff = z_i_next - z_i
        z_i = z_i_next
    h = sqrt(Pos_.x**2 + Pos_.y**2 + z_i**2) - N_i
    lat = asin(sin_lat)
    lon = atan2(Pos_.y, Pos_.x)
    p = sqrt(Pos_.x**2 + Pos_.y**2)
    h = p / cos(lat) - N_i
    return degrees(lat), degrees(lon), h/1000  
    
steps_central_body = []
y_0 = Vector6(Pos_.x, Pos_.y, Pos_.z, Vel_.x, Vel_.y, Vel_.z)
h = Step_Size
w_ = Vector3(0, 0, 72.921151467e-6)
v_r_ = Vel_ - (w_.cross(Pos_))
temp_pos = copy.deepcopy(Pos_)
temp_vel = copy.deepcopy(Vel_)
timestamp = start_time
gah, pos_ECEF_np = rotate_TOD_to_ECEF(start_time, Pos_, Earth_rotation)

steps_central_body.append(Step(0, y_0.x, y_0.y, y_0.z, temp_vel.x, temp_vel.y, temp_vel.z, LATD, LON, ALTD, gah))

for i in range(Step_Num):
    t = h*(i+1)
    timestamp = timestamp + timedelta(seconds=h)
    step = compute_rk_2_body_step(0, Earth_gravitational_parameter, temp_pos, temp_vel, h, y_0, timestamp, 0, 0, 0)
    gah, pos_ECEF_np = rotate_TOD_to_ECEF(timestamp, Vector3(step.x, step.y, step.z), Earth_rotation)
    pos_ECEF = Vector3(pos_ECEF_np[0][0], pos_ECEF_np[1][0], pos_ECEF_np[2][0])
    lat, lon, height = calculate_lat_lon_h(Earth_Eccentricity, pos_ECEF, Earth_radius)
    steps_central_body.append(Step(t, step.x, step.y, step.z, step.xd, step.yd, step.zd, lat, lon, height, gah))
    y_0 = step
    temp_pos = Vector3(step.x, step.y, step.z)
    temp_vel = Vector3(step.xd, step.yd, step.zd)



from IPython.display import Markdown, display

header = "| Step | Time (s) | RK_X (m) | RK_Y (m) | RK_Z (m) | RK_XD (m/s) | RK_YD (m/s) | RK_ZD (m/s) | GAH (deg) | LON (deg) | LAT (deg) | ALT (m) |"
separator = "|------|----------|----------|----------|----------|--------------|--------------|--------------|-----------|-----------|---------|---------|"

rows = []
for i, s in enumerate(steps_central_body):
    row = (
        f"| {i} | {s.T:.2f} | {s.RK_X:.2f} | {s.RK_Y:.2f} | {s.RK_Z:.2f} | "
        f"{s.RK_XD:.2f} | {s.RK_YD:.2f} | {s.RK_ZD:.2f} | {s.GAH:.4f} | "
        f"{s.LON:.4f} | {s.LAT:.4f} | {s.ALTD:.2f} |"
    )
    rows.append(row)

table_md = "\n".join([header, separator] + rows)

display(Markdown(f"### RK4 Integration Steps (State + Geodetic)\n\n{table_md}"))

import matplotlib.pyplot as plt

lats = [s.LAT for s in steps_central_body]
lons = [s.LON for s in steps_central_body]

plt.figure()
plt.scatter(lons, lats)
plt.xlabel("Longitude (deg)")
plt.ylabel("Latitude (deg)")
plt.title("Latitude vs Longitude")
plt.grid()

plt.show()

## Problems 2–3: TOD Input Vector

### Epoch: 3 Jan 2012 02:00:00

### State Vector

| Parameter | Value            | Units  | Parameter | Value      | Units |
|----------|------------------|--------|----------|------------|--------|
| X        | -20129085.474922 | m      | A        | 26561054   | m      |
| Y        | -6489791.087436  | m      | E        | 0.001061   | —      |
| Z        | 16070179.605459  | m      | I        | 54.94887   | deg    |
| XD       | -690.692301      | m/sec  | RAAN     | 50.08203   | deg    |
| YD       | -3158.384437     | m/sec  | WP       | 40.04967   | deg    |
| ZD       | -2133.836396     | m/sec  | NU       | 92.30112   | deg    |

### Geodetic Coordinates

| Parameter | Value       | Units |
|----------|-------------|--------|
| LATD     | 37.259138   | deg    |
| LON      | 65.751825   | deg    |
| ALTD     | 20191.87164 | km     |

- Assume epoch is **UT1**

### Ground Site Location: LION

| Site | Geodetic Latitude (deg) | East Longitude (deg) | Height above reference ellipsoid (m) | Xs (m)      | Ys (m)     | Zs (m)      | Radius (m) |
|------|--------------------------|----------------------|--------------------------------------|-------------|------------|-------------|------------|
| LION | 51.115                   | 359.094              | 146.500                              | 4011670.407 | -63440.560 | 4941699.996 | 6365369.040 |

# Problem 2: For LION and a TOD vector:
- Compute Greenwich Hour Angle at vector epoch
- Compute ECEF satellite vector
- Compute ECEF ground site vector
- Compute relative position between SV and site
- Compute ECEF→Topocentric transformation
- Compute relative position in Topocentric frame
- Compute azimuth and elevation of satellite

In [ ]:
problem_2_epoch = datetime(2012,1,3,2,0,0)
sen_lat = radians(51.115)
sen_lon = radians(359.094)
sen_height = 20191.87164
Pos_2_sat_ = Vector3(-20129085.474922, -6489791.087436, 16070179.605459)
Vel_2_sat_ = Vector3(-690.692301, -3158.384437, -2133.836396)
gha_sat, Pos_2_ECEF_sat_ = rotate_TOD_to_ECEF(problem_2_epoch, Pos_2_sat_, Earth_rotation)
print(f"GAH: {degrees(gah%(2*pi))} deg\n")
Pos_2_sensor_ = Vector3(4011670.406754, -63440.560202, 4941699.996211) # already ECEF
print(f"ECEF Sat Vector: \n{Pos_2_ECEF_sat_}\n")
print(f"Relative Distance Between Sat and Sensor in ECEF: \n{Pos_2_ECEF_sat_ - Pos_2_sensor_.get_np_vector()}\n")
R_ECEF_TOPOCENTRIC = np.array([
    [-sin(sen_lon), cos(sen_lon), 0],
    [-sin(sen_lat)*cos(sen_lon), -sin(sen_lat)*sin(sen_lon), cos(sen_lat)],
    [ cos(sen_lat)*cos(sen_lon), cos(sen_lat)*sin(sen_lon), sin(sen_lat)]
])
print(f"ECEF to Topocentric rotation matrix: \n{R_ECEF_TOPOCENTRIC}\n")
TOCOCENTRIC_Pos_sat_ = R_ECEF_TOPOCENTRIC @ Pos_2_ECEF_sat_
TOPOCENTRIC_Pos_sen_ = R_ECEF_TOPOCENTRIC @ Pos_2_sensor_.get_np_vector()
sen_to_sat_ = TOCOCENTRIC_Pos_sat_ - TOPOCENTRIC_Pos_sen_
print(f"Relative Distance Between Sat and Sensor in Topocentric: \n{sen_to_sat_}\n")
Az = atan2(sen_to_sat_[0][0], sen_to_sat_[1][0])
El = atan(sen_to_sat_[2][0]/sqrt(sen_to_sat_[0][0]**2 + sen_to_sat_[1][0]**2))
print(f"Azimuth: {degrees(Az)}")
print(f"Elevation: {degrees(El)}")

# Problem 3: For same site and Problem 2 TOD vector
- Compute TOD ground site vector
- Compute range using instantaneous range method
- Using light-time algorithm, compute time to traverse SV → RCVR leg  
  and add Δτ/2 to get one-way range
  - Let Δτ = 10⁻⁶ sec

- Compute difference between instantaneous and light-time range

In [ ]:
light_speed = 299792458.0
gah, Pos_2_sensor_TOD_ = rotate_TOD_to_ECEF(problem_2_epoch, Pos_2_sensor_, Earth_rotation, True)
print(f"TOD ground site vector: \n{Pos_2_sensor_TOD_}\n")
print(f"Sensor to Satellite Vector: {Vector3(sen_to_sat_[0][0], sen_to_sat_[1][0], sen_to_sat_[2][0]).magnitude()} m")

#instantaneous
sen_TOD = Vector3(float(Pos_2_sensor_TOD_[0][0]), float(Pos_2_sensor_TOD_[1][0]), float(Pos_2_sensor_TOD_[2][0]))
pest = Vector3(sen_to_sat_[0][0], sen_to_sat_[1][0], sen_to_sat_[2][0]).magnitude()
del_t_est = Vector3(sen_to_sat_[0][0], sen_to_sat_[1][0], sen_to_sat_[2][0]).magnitude()/light_speed
total_delta_t = del_t_est
initial_delta_t =0
tv = problem_2_epoch - timedelta(seconds=del_t_est)
y_0 = Vector6(Pos_2_sat_.x, Pos_2_sat_.y, Pos_2_sat_.z, Vel_2_sat_.x, Vel_2_sat_.y, Vel_2_sat_.z)
step = compute_rk_2_body_step(0, Earth_gravitational_parameter, Pos_2_sat_, Vel_2_sat_, -del_t_est, y_0, problem_2_epoch, 0, 0, 0)
rtv = Vector3(float(step.x), float(step.y), float(step.z))
p = (sen_TOD - rtv).magnitude()
print(f"Instantaneous One-Way Range: {p} m")

# light time algorithm for receive
difference = 100000.0
last_dt = 0.0
rx = Pos_2_sat_
step_num = 1

class leg_step():
    def __init__(self, step, range, prop_time, difference):
        self.step = step
        self.range = range
        self.prop_time = prop_time
        self.difference = difference

step_data = []
while abs(difference) > 0.0000001:
    dr = rx - sen_TOD
    rho = dr.magnitude()
    dt = rho / light_speed
    difference = abs(dt - last_dt)
    last_dt = dt
    y_0 = Vector6(Pos_2_sat_.x, Pos_2_sat_.y, Pos_2_sat_.z, Vel_2_sat_.x, Vel_2_sat_.y, Vel_2_sat_.z)
    step = compute_rk_2_body_step(0, Earth_gravitational_parameter, Pos_2_sat_, Vel_2_sat_, -dt, y_0, problem_2_epoch, 0, 0, 0)
    rx = Vector3(float(step.x), float(step.y), float(step.z))
    step_data.append(leg_step(step_num, rho, dt, difference))
    step_num += 1

header = "| Step | Range(m) | PropTime | Difference |"
separator = "|----------|-------|-------|-------|"

rows = []
for step in step_data:
    row = f"| {step.step} | {step.range} | {step.prop_time} | {step.difference} |"
    rows.append(row)

table_md = "\n".join([header, separator] + rows)
display(Markdown(f"### RK4 Integration Steps (2 body + Geopotential)\n\n{table_md}"))

light_time_rcv_leg = step_data[len(step_data) - 1].range
print(f"Light Time Algorithm One-Way Range: {light_time_rcv_leg} m")

delta_tao_half = (1e-6)/2
print(f"XPDR Bias: {delta_tao_half*light_speed} m")
range_plus_compute_time_range = delta_tao_half*light_speed + step_data[len(step_data) - 1].range
print(f"Instantaneous verses light time range diff: {light_time_rcv_leg - p} m")